In [ ]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [ ]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:00<00:00, 164MB/s] 


In [ ]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [ ]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [ ]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [ ]:
import yfinance as yf
df = yf.download(tickers, period="3mo")
df.head()

/tmp/ipykernel_1238/88378548.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="3mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2026-04-27  115.398041  267.363647  195.985016  141.059998  92.142815   
2026-04-28  114.649483  270.460815  196.292816  139.039993  93.195305   
2026-04-29  111.086334  269.921326  202.448990  140.279999  90.683220   
2026-04-30  115.328178  271.100250  209.826492  140.360001  90.147041   
2026-05-01  114.300148  279.882141  205.139847  141.660004  88.826462   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2026-04-27  96.349998  174.867355  239.309998  391.517578  70.077965  ...   
2026-04-28  97.059998  175.638092  243.199997  382.213074  72.323166  ...   
2026-04-29  92.720001  178.118256  243.570007  388.246552  73.783531  ...   
2026-04-30  94.459999  176.586685  246.100006  401.161194  74.051765  ...   
2026-05-01  93.820000  177.693375  250.710007  396.603668  74.449150  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2026-04-27  3505500  1175000  3892000  13940800  2983600  3991200  2854500   
2026-04-28  3987800  1083900  3790500  15173600  3360600  4507400  3013400   
2026-04-29  5619400  1352700  3719000  14669100  2660800  4924800  3177300   
2026-04-30  8281500  1086800  9105500  22885900  2543000  4106200  1878700   
2026-05-01  7384500  1466300  5345900  15458200  1872000  3808600  1589100   

Price                                  
Ticker          ZBH     ZBRA      ZTS  
Date                                   
2026-04-27  2503800   738700  3160300  
2026-04-28  9823700   760600  2972500  
2026-04-29  5242800   891400  3395900  
2026-04-30  4169700  1016000  4297200  
2026-05-01  2724100   655900  3204000  

[5 rows x 2515 columns]

In [ ]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)


/tmp/ipykernel_1238/3524405700.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
501,2026-04-28,A,114.649483,-0.648675
502,2026-04-28,AAPL,270.460815,1.158410
503,2026-04-28,ABBV,196.292816,0.157053
504,2026-04-28,ABNB,139.039993,-1.432018
505,2026-04-28,ABT,93.195305,1.142238
506,2026-04-28,ACGL,97.059998,0.736896
507,2026-04-28,ACN,175.638092,0.440755
508,2026-04-28,ADBE,243.199997,1.625506
509,2026-04-28,ADI,382.213074,-2.376523
510,2026-04-28,ADM,72.323166,3.203862


In [ ]:
len(fdf)

30628

In [ ]:
df = fdf

df['lag 1'] = df['return'].shift(1)
df['lag 2'] = df['return'].shift(2)
df['lag 3'] = df['return'].shift(3)
df['lag 5'] = df['return'].shift(5)

df.head(25)



,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5
501,2026-04-28,A,114.649483,-0.648675,NaN,NaN,NaN,NaN
502,2026-04-28,AAPL,270.460815,1.158410,-0.648675,NaN,NaN,NaN
503,2026-04-28,ABBV,196.292816,0.157053,1.158410,-0.648675,NaN,NaN
504,2026-04-28,ABNB,139.039993,-1.432018,0.157053,1.158410,-0.648675,NaN
505,2026-04-28,ABT,93.195305,1.142238,-1.432018,0.157053,1.158410,NaN
506,2026-04-28,ACGL,97.059998,0.736896,1.142238,-1.432018,0.157053,-0.648675
507,2026-04-28,ACN,175.638092,0.440755,0.736896,1.142238,-1.432018,1.158410
508,2026-04-28,ADBE,243.199997,1.625506,0.440755,0.736896,1.142238,0.157053
509,2026-04-28,ADI,382.213074,-2.376523,1.625506,0.440755,0.736896,-1.432018
510,2026-04-28,ADM,72.323166,3.203862,-2.376523,1.625506,0.440755,1.142238


In [ ]:
df = df.drop(columns=['lag 1', 'lag 2', 'lag 3', 'lag 5'])
df.head()




,date,ticker,close,return
501,2026-04-28,A,114.649483,-0.648675
502,2026-04-28,AAPL,270.460815,1.158410
503,2026-04-28,ABBV,196.292816,0.157053
504,2026-04-28,ABNB,139.039993,-1.432018
505,2026-04-28,ABT,93.195305,1.142238


In [ ]:
df['lag 1'] = df.groupby('ticker')['return'].shift(1)
df['lag 2'] = df.groupby('ticker')['return'].shift(2)
df['lag 3'] = df.groupby('ticker')['return'].shift(3)
df['lag 5'] = df.groupby('ticker')['return'].shift(5)


df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5
501,2026-04-28,A,114.649483,-0.648675,NaN,NaN,NaN,NaN
502,2026-04-28,AAPL,270.460815,1.158410,NaN,NaN,NaN,NaN
503,2026-04-28,ABBV,196.292816,0.157053,NaN,NaN,NaN,NaN
504,2026-04-28,ABNB,139.039993,-1.432018,NaN,NaN,NaN,NaN
505,2026-04-28,ABT,93.195305,1.142238,NaN,NaN,NaN,NaN


In [ ]:
df['volatility 5'] = df['return'].rolling(window=5).std()
df['volatility 10'] = df['return'].rolling(window=10).std()
df['volatility 20'] = df['return'].rolling(window=20).std()

df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5,volatility 5,volatility 10,volatility 20
501,2026-04-28,A,114.649483,-0.648675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502,2026-04-28,AAPL,270.460815,1.158410,NaN,NaN,NaN,NaN,NaN,NaN,NaN
503,2026-04-28,ABBV,196.292816,0.157053,NaN,NaN,NaN,NaN,NaN,NaN,NaN
504,2026-04-28,ABNB,139.039993,-1.432018,NaN,NaN,NaN,NaN,NaN,NaN,NaN
505,2026-04-28,ABT,93.195305,1.142238,NaN,NaN,NaN,NaN,1.130743,NaN,NaN


In [ ]:
#simple moving averages

df['sma10'] = df['close'].rolling(window=10).mean()
df['sma20'] = df['close'].rolling(window=20).mean()

df['sma1/2'] = df['sma10']/df['sma20']

df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5,volatility 5,volatility 10,volatility 20,sma10,sma20,sma1/2
501,2026-04-28,A,114.649483,-0.648675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502,2026-04-28,AAPL,270.460815,1.158410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
503,2026-04-28,ABBV,196.292816,0.157053,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
504,2026-04-28,ABNB,139.039993,-1.432018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
505,2026-04-28,ABT,93.195305,1.142238,NaN,NaN,NaN,NaN,1.130743,NaN,NaN,NaN,NaN,NaN


In [ ]:
df['tomorrows returns'] = df.groupby('ticker')['return'].shift(-1)

df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5,volatility 5,volatility 10,volatility 20,sma10,sma20,sma1/2,tomorrows returns
501,2026-04-28,A,114.649483,-0.648675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.107863
502,2026-04-28,AAPL,270.460815,1.158410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.199471
503,2026-04-28,ABBV,196.292816,0.157053,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.136220
504,2026-04-28,ABNB,139.039993,-1.432018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.891834
505,2026-04-28,ABT,93.195305,1.142238,NaN,NaN,NaN,NaN,1.130743,NaN,NaN,NaN,NaN,NaN,-2.695506


In [ ]:
df = df.dropna()
df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5,volatility 5,volatility 10,volatility 20,sma10,sma20,sma1/2,tomorrows returns
3006,2026-05-05,A,117.304375,4.769121,-2.043307,-0.891396,3.818511,-0.648675,2.863997,2.110227,1.664314,121.171007,138.103330,0.877394,0.136134
3007,2026-05-05,AAPL,283.918427,2.655069,-1.181560,3.239352,0.436766,1.158410,2.734033,2.198027,1.809221,139.245950,141.479190,0.984215,1.171792
3008,2026-05-05,ABBV,204.653305,-0.984825,0.755085,-2.233581,3.644129,0.157053,2.825354,2.145264,1.776031,151.655041,147.775361,1.026254,-0.523995
3009,2026-05-05,ABNB,139.729996,0.626527,-1.976566,0.926192,0.057030,-1.432018,2.558305,2.145555,1.790194,150.362653,143.418518,1.048419,0.107356
3010,2026-05-05,ABT,86.552673,-0.422671,-2.146210,-1.464917,-0.591265,1.142238,2.372156,2.144487,1.789918,147.578772,144.003291,1.024829,-0.998040


In [ ]:
len(df)

27610

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz

df['direction'] = (df['tomorrows returns'] > 0).astype(int)

In [ ]:
df.head()

,date,ticker,close,return,lag 1,lag 2,lag 3,lag 5,volatility 5,volatility 10,volatility 20,sma10,sma20,sma1/2,tomorrows returns,direction
3006,2026-05-05,A,117.304375,4.769121,-2.043307,-0.891396,3.818511,-0.648675,2.863997,2.110227,1.664314,121.171007,138.103330,0.877394,0.136134,1
3007,2026-05-05,AAPL,283.918427,2.655069,-1.181560,3.239352,0.436766,1.158410,2.734033,2.198027,1.809221,139.245950,141.479190,0.984215,1.171792,1
3008,2026-05-05,ABBV,204.653305,-0.984825,0.755085,-2.233581,3.644129,0.157053,2.825354,2.145264,1.776031,151.655041,147.775361,1.026254,-0.523995,0
3009,2026-05-05,ABNB,139.729996,0.626527,-1.976566,0.926192,0.057030,-1.432018,2.558305,2.145555,1.790194,150.362653,143.418518,1.048419,0.107356,1
3010,2026-05-05,ABT,86.552673,-0.422671,-2.146210,-1.464917,-0.591265,1.142238,2.372156,2.144487,1.789918,147.578772,144.003291,1.024829,-0.998040,0


In [ ]:
features = ['close', 'return', 'lag 1', 'lag 2', 'lag 3', 'lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'sma10' ,'sma20', 'sma1/2']
x = df[features]
y = df['direction']



In [ ]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
from sklearn.metrics import accuracy_score
tree = DecisionTreeClassifier(min_samples_leaf = 5, random_state=72)

tree.fit(x_train, y_train)
predictions = tree.predict(x_test)

score = accuracy_score(y_test, predictions)
score


0.4961970300615719

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(25,12))

plot_tree(tree, feature_names=features, class_names=["down","up"], filled=True, rounded=True)
plt.show()

In [ ]:
def get_tree():
  rng = np.random.RandomState(72)
  n_rows = len(x_train)
  ri = rng.choice(n_rows, size = n_rows, replace=True)

  sx = x_train.iloc[ri]
  sy = y_train.iloc[ri]
  return DecisionTreeClassifier(min_samples_leaf = 5, max_features = .5).fit(sx, sy)





In [ ]:
forest = [get_tree() for t in range(100)]

In [ ]:
from sklearn.metrics import mean_absolute_error
all_probs = [t.predict(x_test) for t in forest]
avg_probs = np.stack(all_probs).mean(0)

score = mean_absolute_error(y_test, avg_probs)
score

0.5028033321260413

In [ ]:
# Set up a massive canvas with a 10x10 grid of subplots
fig, axes = plt.subplots(nrows=10, ncols=10, figsize=(50, 50))

# Flatten the 10x10 grid into a single list so we can loop through it
axes = axes.flatten()

print("Planting the visual forest... this might take a minute...")

# Loop through all 100 trees in your list
for i, tree in enumerate(forest):
    # Plot each tree on its own specific mini-canvas (ax)
    plot_tree(
        tree,
        ax=axes[i],               # Tells it which subplot to draw on
        filled=True,
        rounded=True,
        # We leave out feature_names and class_names here because
        # the text would just turn into black ink blobs anyway
    )
    axes[i].set_title(f"Tree {i}", fontsize=14)

# Cleans up the spacing between the 100 plots
plt.tight_layout()
plt.show()